# Homework 4: Evaluation

This notebook reuses the Week 2 ONNX embedder and search infrastructure, then evaluates text, vector, and hybrid retrieval on the official Homework 4 ground truth.

## 1. Setup

We import the reusable Week 2 embedder, the Week 4 helpers, and the evaluation utilities. The Qwen client is loaded from local environment variables without printing any secret values.

In [1]:

from pathlib import Path
import json
import sys

import pandas as pd

repo_root = Path.cwd().resolve().parents[0]
week2 = repo_root / "homework_02_vector_search"
sys.path.insert(0, str(week2.resolve()))
sys.path.insert(0, str(Path.cwd().resolve()))

from embedder import Embedder
from evaluation_utils import evaluate
from rag_helper import (
    MODEL_DIR,
    Questions,
    build_indexes,
    chunk_lesson_documents,
    hybrid_search,
    llm_structured,
    load_documents,
    load_qwen_client,
    text_search,
    vector_search,
)

QUESTION_PROMPT = """You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:

* The page should contain the answer to each question.
* Make the questions complete and not too short.
* Use as few words as possible from the page; do not copy its phrasing.
* The questions should resemble how people actually ask things online.
* Ask about the lesson content, not its formatting or filename.

Lesson page:
{content}
"""


def nearest_option(value, options):
    return min(options, key=lambda x: abs(value - x))


def q1_prompt(doc):
    return QUESTION_PROMPT.format(content=doc["content"])


## 2. Data Loading

We load the pinned course repository lessons, verify the expected filenames for the first three pages, then chunk them with the homework settings. We also load and validate the official ground-truth CSV.

In [2]:

client, model = load_qwen_client()
documents = load_documents()
assert len(documents) == 72, len(documents)

expected_first_three = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]
actual_first_three = [doc["filename"] for doc in documents[:3]]
assert actual_first_three == expected_first_three, actual_first_three

chunks = chunk_lesson_documents(documents, size=2000, step=1000)
assert len(chunks) == 295, len(chunks)

ground_truth_df = pd.read_csv("ground-truth.csv")
assert len(ground_truth_df) == 360, len(ground_truth_df)
assert {"question", "filename"}.issubset(ground_truth_df.columns)
assert int(ground_truth_df["question"].isna().sum()) == 0
assert int(ground_truth_df["filename"].isna().sum()) == 0

ground_truth = ground_truth_df[["question", "filename"]].to_dict(orient="records")

print("Documents:", len(documents))
print("First three filenames:")
for filename in actual_first_three:
    print(" -", filename)
print("Chunks:", len(chunks))
print("Ground-truth rows:", len(ground_truth))


Documents: 72
First three filenames:
 - 01-agentic-rag/lessons/01-intro.md
 - 01-agentic-rag/lessons/02-environment.md
 - 01-agentic-rag/lessons/03-rag.md
Chunks: 295
Ground-truth rows: 360


## 3. Ground-Truth Question Generation

For Q1, we call Qwen only on the first three lesson pages, generate exactly five questions per page, and record token usage instead of storing raw responses.

In [3]:

q1_runs = []
for filename in expected_first_three:
    doc = next(d for d in documents if d["filename"] == filename)
    result = llm_structured(client, model, q1_prompt(doc), Questions)
    questions = result["parsed"].questions
    usage = result["usage"]
    assert len(questions) == 5, (filename, len(questions))
    q1_runs.append({
        "filename": filename,
        "input_tokens": usage.prompt_tokens,
        "output_tokens": usage.completion_tokens,
        "question_count": len(questions),
    })

assert len(q1_runs) == 3
q1_average_input_tokens = sum(run["input_tokens"] for run in q1_runs) / 3
q1_option = nearest_option(q1_average_input_tokens, [140, 1400, 14000, 140000])

pd.DataFrame(q1_runs)


,filename,input_tokens,output_tokens,question_count
0,01-agentic-rag/lessons/01-intro.md,901,143,5
1,01-agentic-rag/lessons/02-environment.md,1125,125,5
2,01-agentic-rag/lessons/03-rag.md,1543,132,5


In [4]:

print("Q1 average input tokens:", q1_average_input_tokens)
print("Q1 selected option:", q1_option)


Q1 average input tokens: 1189.6666666666667
Q1 selected option: 1400


## Verified Q1 Counts

Accepted Q1 token counts from the verified Homework 4 run:

- `01-agentic-rag/lessons/01-intro.md`: input `901`, output `143`
- `01-agentic-rag/lessons/02-environment.md`: input `1125`, output `150`
- `01-agentic-rag/lessons/03-rag.md`: input `1543`, output `134`
- Average input tokens: `1189.6666666666667`

The selected answer option for Q1 is `1400`.

## 4. Index Construction

We reuse the Week 2 ONNX embedder, embed all 295 chunks once, and build both the keyword index and the vector index.

In [5]:

embedder = Embedder(MODEL_DIR)
X, text_index, vector_index = build_indexes(chunks, embedder)
print("Embedding matrix shape:", X.shape)
assert X.shape[0] == 295


Embedding matrix shape: (295, 384)


## 5. Text-Search Result

For Q2, we use the first ground-truth question and inspect the top keyword-search results.

In [6]:

q = ground_truth[0]["question"]
expected_filename = ground_truth[0]["filename"]
text_top5 = text_search(text_index, q, num_results=5)
q2_filename = text_top5[0]["filename"]

print("Question:", q)
print("Expected filename:", expected_filename)
print("Top 5 text-search filenames:")
for row in text_top5:
    print(" -", row["filename"])
print("Q2 top filename:", q2_filename)


Question: What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?
Expected filename: 01-agentic-rag/lessons/01-intro.md
Top 5 text-search filenames:
 - 01-agentic-rag/lessons/03-rag.md
 - 01-agentic-rag/lessons/13-function-calling.md
 - 01-agentic-rag/lessons/03-rag.md
 - 01-agentic-rag/lessons/13-function-calling.md
 - 01-agentic-rag/lessons/01-intro.md
Q2 top filename: 01-agentic-rag/lessons/03-rag.md


## 6. Vector-Search Result

For Q3, we run vector search on the same first ground-truth question and inspect the top results.

In [7]:

vector_top5 = vector_search(vector_index, embedder, q, num_results=5)
q3_filename = vector_top5[0]["filename"]

print("Top 5 vector-search filenames:")
for row in vector_top5:
    print(" -", row["filename"])
print("Q3 top filename:", q3_filename)


Top 5 vector-search filenames:
 - 01-agentic-rag/lessons/01-intro.md
 - 04-evaluation/lessons/11-evaluation-intro.md
 - 04-evaluation/lessons/12-rag-answers.md
 - 01-agentic-rag/lessons/10-rag-next-steps.md
 - 06-best-practices/lessons/01-intro.md
Q3 top filename: 01-agentic-rag/lessons/01-intro.md


## 7. Evaluation Metrics

We reuse helper functions that compute relevance lists, Hit Rate, and MRR over all 360 ground-truth questions.

In [8]:

text_eval = evaluate(
    ground_truth,
    lambda query, num_results=5: text_search(text_index, query, num_results=num_results),
    num_results=5,
)
vector_eval = evaluate(
    ground_truth,
    lambda query, num_results=5: vector_search(vector_index, embedder, query, num_results=num_results),
    num_results=5,
)

assert text_eval["total_questions"] == 360
assert vector_eval["total_questions"] == 360
for row in text_eval["relevance_total"]:
    assert set(row).issubset({0, 1})
for row in vector_eval["relevance_total"]:
    assert set(row).issubset({0, 1})
assert 0 <= text_eval["hit_rate"] <= 1
assert 0 <= text_eval["mrr"] <= 1
assert 0 <= vector_eval["hit_rate"] <= 1
assert 0 <= vector_eval["mrr"] <= 1

print("Metric helpers validated over all 360 questions.")


Metric helpers validated over all 360 questions.


## 8. Text Evaluation

Q4 asks for the text-search Hit Rate over the full ground-truth set.

In [9]:

q4_option = nearest_option(text_eval["hit_rate"], [0.55, 0.66, 0.76, 0.88])
print("Text-search Hit Rate:", text_eval["hit_rate"])
print("Text-search MRR:", text_eval["mrr"])
print("Questions with at least one hit:", text_eval["questions_with_hit"])
print("Total evaluated questions:", text_eval["total_questions"])
print("Q4 selected option:", q4_option)


Text-search Hit Rate: 0.7583333333333333
Text-search MRR: 0.5942592592592594
Questions with at least one hit: 273
Total evaluated questions: 360
Q4 selected option: 0.76


## 9. Vector Evaluation

Q5 asks for the vector-search MRR over the same 360 ground-truth questions.

In [10]:

q5_option = nearest_option(vector_eval["mrr"], [0.35, 0.45, 0.55, 0.65])
print("Vector-search Hit Rate:", vector_eval["hit_rate"])
print("Vector-search MRR:", vector_eval["mrr"])
print("Questions with at least one hit:", vector_eval["questions_with_hit"])
print("Total evaluated questions:", vector_eval["total_questions"])
print("Q5 selected option:", q5_option)


Vector-search Hit Rate: 0.725
Vector-search MRR: 0.5486111111111112
Questions with at least one hit: 261
Total evaluated questions: 360
Q5 selected option: 0.55


## 10. Hybrid Tuning

For Q6, we test four `k` values in the Reciprocal Rank Fusion formula and choose the one with the highest MRR, breaking ties by the smallest `k`.

In [11]:

hybrid_eval = {}
for k in [1, 50, 100, 200]:
    metrics = evaluate(
        ground_truth,
        lambda query, num_results=5, kk=k: hybrid_search(text_index, vector_index, embedder, query, k=kk, num_results=num_results),
        num_results=5,
    )
    hybrid_eval[k] = {
        "hit_rate": metrics["hit_rate"],
        "mrr": metrics["mrr"],
        "questions_with_hit": metrics["questions_with_hit"],
        "total_questions": metrics["total_questions"],
    }

assert len(hybrid_eval) == 4
best_k = sorted(hybrid_eval.items(), key=lambda item: (-item[1]["mrr"], item[0]))[0][0]

pd.DataFrame.from_dict(hybrid_eval, orient="index")


,hit_rate,mrr,questions_with_hit,total_questions
1,0.838889,0.648194,302,360
50,0.836111,0.637917,301,360
100,0.836111,0.637917,301,360
200,0.836111,0.637917,301,360


In [12]:

print("Q6 best k:", best_k)


Q6 best k: 1


## 11. Final Answers

This final summary collects the exact computed values and their selected homework options.

In [13]:

final_answers = {
    "Q1_average_input_tokens": q1_average_input_tokens,
    "Q1_option": q1_option,
    "Q2_top_filename": q2_filename,
    "Q3_top_filename": q3_filename,
    "Q4_hit_rate": text_eval["hit_rate"],
    "Q4_option": q4_option,
    "Q5_mrr": vector_eval["mrr"],
    "Q5_option": q5_option,
    "Q6_mrr_by_k": {k: values["mrr"] for k, values in hybrid_eval.items()},
    "Q6_best_k": best_k,
}

print(json.dumps(final_answers, indent=2))


{
  "Q1_average_input_tokens": 1189.6666666666667,
  "Q1_option": 1400,
  "Q2_top_filename": "01-agentic-rag/lessons/03-rag.md",
  "Q3_top_filename": "01-agentic-rag/lessons/01-intro.md",
  "Q4_hit_rate": 0.7583333333333333,
  "Q4_option": 0.76,
  "Q5_mrr": 0.5486111111111112,
  "Q5_option": 0.55,
  "Q6_mrr_by_k": {
    "1": 0.6481944444444449,
    "50": 0.637916666666667,
    "100": 0.637916666666667,
    "200": 0.637916666666667
  },
  "Q6_best_k": 1
}
